# Daily Climate-Health Models: Explanatory and Predictive

Two tracks, sharing the same daily panel dataset.

**Track A — Explanatory (DLNM, no AR)**
Quasi-Poisson GLM with seasonal spline + DOW dummies + DLNM cross-basis.
Tests whether climate affects daily diarrheal cases, independent of AR structure.
Fits main effect and infrastructure-quality interaction.

**Track B — Predictive (multi-horizon)**
Evaluates climate's contribution at forecast horizons 1, 7, 14, 21, 28 days.
Compares AR+Seasonal+Climate vs AR+Seasonal vs Climate-only (no AR) vs Seasonal-only.
The climate-only row is the key early-warning system benchmark.

**Input:** `{OUT_ROOT}/modeling/{NETWORK}_{HSA_MODE}_daily_modeling_dataset_{BOUNDARY_VERSION}.csv`


In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
# Choose which HSA boundary bundle to use. Must match a bundle from HSA_FINAL.ipynb.
#   'v6'  — original greedy algorithm (no post-selection corrections)
#   'v7'  — + anchor upgrade/demotion + major-orphan promotion
#   'v8'  — + satellite bubble boundaries
BOUNDARY_VERSION = "v8"         # change as needed

NETWORK          = "INF"        # change as needed (INF or NCD)
HSA_MODE         = "footprint"  # change as needed
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
# Setup
import sys
from pathlib import Path
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Import cross-basis functions from the local dlnm/ package
sys.path.insert(0, str(Path(".")))
from dlnm.dlnm_crossbasis import ns_basis, build_crossbasis, cumulative_rr

OUT_ROOT  = Path(os.environ.get("HSA_OUT_DIR", os.environ.get("PIPELINE_OUT_DIR", "out")))
DATA_FILE = OUT_ROOT / "modeling" / f"{NETWORK}_{HSA_MODE}_daily_modeling_dataset_{BOUNDARY_VERSION}.csv"
OUT_DIR   = OUT_ROOT / "modeling" / f"daily_models_{BOUNDARY_VERSION}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries loaded")
print(f"Boundary version: {BOUNDARY_VERSION}")
print(f"Network:          {NETWORK}")
print(f"Data file: {DATA_FILE}")
print(f"Output: {OUT_DIR.resolve()}")


In [3]:
# Load data
df = pd.read_csv(DATA_FILE, parse_dates=["date"])
df = df.sort_values(["hsa_id", "date"]).reset_index(drop=True)
print(f"Loaded: {df.shape}  HSAs={df['hsa_id'].nunique()}  "
      f"dates={df['date'].dt.date.min()} to {df['date'].dt.date.max()}")

# HSAs with mean > 1 case/day (avoid sparse fitting)
hsa_means = df.groupby("hsa_id")["diarrheal_count"].mean()
low_count_hsas = hsa_means[hsa_means < 1.0].index.tolist()
if low_count_hsas:
    print(f"Excluding {len(low_count_hsas)} low-count HSAs: {low_count_hsas}")

df_full = df[~df["hsa_id"].isin(low_count_hsas)].copy().reset_index(drop=True)
print(f"Analysis dataset: {df_full.shape}  HSAs={df_full['hsa_id'].nunique()}")


Loaded: (10716, 178)  HSAs=19  dates=2022-07-15 to 2024-01-31
Excluding 3 low-count HSAs: ['Khazzan_Primary_Center', 'Mabroukeh_Primary_Center', 'Prince_Hashem_Primary_Center']
Analysis dataset: (9024, 178)  HSAs=16


In [4]:
# ─── Base predictors (shared across both tracks) ─────────────────────────────

outcome    = df_full["diarrheal_count"].values.astype(float)
infra      = df_full["infra_quality"].fillna(df_full["infra_quality"].mean()).values
infra_c    = infra - infra.mean()

# HSA fixed effects
hsa_dummies = pd.get_dummies(df_full["hsa_id"], drop_first=True, dtype=float)

# Seasonal spline: ~5 knots per year of study
n_days = int(df_full["day_of_study"].max()) + 1
n_int  = max(5, n_days // 90)          # roughly one interior knot per quarter
spline_t, spline_knots = ns_basis(df_full["day_of_study"].values, n_interior_knots=n_int)

# DOW dummies (reference = Monday/0)
dow_dummies = pd.get_dummies(df_full["day_of_week"],
                             prefix="dow", drop_first=True, dtype=float)


# Calendar indicators (validated against raw attendance data):
#   is_friday    — Friday 54% of uniform attendance; captured by DOW dummy but
#                  included as explicit column for easy interpretation
#   is_ramadan   — 27% reduction in daily visits during Ramadan
#   is_eid_fitr  — multi-day drop after Ramadan (Eid al-Fitr)
#   is_eid_adha  — sharp 1-2 day drop for Eid al-Adha
# Note: Saturday is NOT a low-attendance day in Jordan (100% of uniform).
#       Other secular holidays show inconsistent signals and are excluded.
for col in ["is_ramadan", "is_eid_fitr", "is_eid_adha"]:
    if col not in df_full.columns:
        df_full[col] = 0
        print(f"  WARNING: {col} not found, set to 0")

calendar_X = np.c_[
    df_full["is_ramadan"].values.astype(float),
    df_full["is_eid_fitr"].values.astype(float),
    df_full["is_eid_adha"].values.astype(float),
]

base_X = np.c_[
    hsa_dummies.values,
    spline_t,
    dow_dummies.values,   # includes Friday dummy (day_of_week==4)
    calendar_X,
]
print(f"Base design matrix: {base_X.shape}")
print(f"  HSA FE:     {hsa_dummies.shape[1]}")
print(f"  Spline:     {spline_t.shape[1]} (df={n_int+1})")
print(f"  DOW:        {dow_dummies.shape[1]} (Fri= {int((df_full['day_of_week']==4).sum())} days)")
print(f"  Ramadan:    {int(df_full['is_ramadan'].sum())} HSA-days")
print(f"  Eid Fitr:   {int(df_full['is_eid_fitr'].sum())} HSA-days")
print(f"  Eid Adha:   {int(df_full['is_eid_adha'].sum())} HSA-days")


Base design matrix: (9024, 31)
  HSA FE:     15
  Spline:     7 (df=7)
  DOW:        6 (Fri= 1296 days)
  Ramadan:    480 HSA-days
  Eid Fitr:   48 HSA-days
  Eid Adha:   32 HSA-days


## Track A — Explanatory DLNM

In [5]:
# ─── Track A: precipitation cross-basis ──────────────────────────────────────
# Lag 0 = same-day precip; lag 1-14 = prior days
lag_vals = np.arange(0, 15, dtype=float)   # 0..14

PRECIP_VARS = ["P_precip"] + [f"P_precip_lag{k}" for k in range(1, 15)]
Q_precip = df_full[PRECIP_VARS].values  # shape (n_obs, 15)

# Knot strategy: 80th pct of non-zero values
all_p = Q_precip.flatten()
nonzero_p = all_p[all_p > 0]
zero_frac  = (all_p == 0).mean()
int_knot_p = np.percentile(nonzero_p, 80) if zero_frac > 0.3 else np.percentile(all_p, 50)
exp_all_knots_p = np.array([all_p.min(), int_knot_p, all_p.max()])

lag_int_knots = np.array([3.0, 7.0])
lag_all_knots = np.array([lag_vals[0], lag_int_knots[0], lag_int_knots[1], lag_vals[-1]])

CB_precip, _, cb_meta = build_crossbasis(
    Q_precip,
    exp_n_int=1,
    lag_n_int=2,
    exp_all_knots=exp_all_knots_p,
    lag_all_knots=lag_all_knots,
    lag_values=lag_vals,
)
print(f"Precipitation cross-basis: {CB_precip.shape}")
print(f"  Zero fraction: {zero_frac:.1%}")
print(f"  Interior knot: {int_knot_p:.2f} mm")
print(f"  Lag knots: {lag_all_knots}")


Precipitation cross-basis: (9024, 6)
  Zero fraction: 87.3%
  Interior knot: 9.51 mm
  Lag knots: [ 0.  3.  7. 14.]


In [6]:
# ─── Fit quasi-Poisson models ─────────────────────────────────────────────────

def fit_qp(X, y):
    m = sm.GLM(y, sm.add_constant(X.astype(float), has_constant="add"),
               family=sm.families.Poisson())
    return m.fit(scale="X2")

def ftest(res_r, res_f):
    dD  = res_r.deviance - res_f.deviance
    ddf = int(round(res_r.df_resid - res_f.df_resid))
    if ddf <= 0:
        return np.nan, np.nan
    F = (dD / ddf) / res_f.scale
    p = 1 - stats.f.cdf(F, ddf, res_f.df_resid)
    return float(F), float(p)

CB_x_infra = CB_precip * infra_c[:, None]

print("Fitting base model...")
res_base  = fit_qp(pd.DataFrame(base_X), outcome)

print("Fitting main-effect model (base + CB_precip)...")
res_main  = fit_qp(np.c_[base_X, CB_precip], outcome)

print("Fitting interaction model (base + CB_precip + CB×infra)...")
res_inter = fit_qp(np.c_[base_X, CB_precip, CB_x_infra], outcome)

F_main, p_main = ftest(res_base,  res_main)
F_int,  p_int  = ftest(res_main,  res_inter)

print()
print("═" * 55)
print("Track A: Precipitation DLNM results")
print("═" * 55)
print(f"  φ (dispersion):              {res_inter.scale:.2f}")
print(f"  Main effect F-test:          F={F_main:.3f}  p={p_main:.4f}")
print(f"  Sanitation interaction:      F={F_int:.3f}   p={p_int:.4f}")


Fitting base model...
Fitting main-effect model (base + CB_precip)...
Fitting interaction model (base + CB_precip + CB×infra)...

═══════════════════════════════════════════════════════
Track A: Precipitation DLNM results
═══════════════════════════════════════════════════════
  φ (dispersion):              3.13
  Main effect F-test:          F=1.734  p=0.1767
  Sanitation interaction:      F=5.376   p=0.0046


In [7]:
# ─── Cumulative RR plot ───────────────────────────────────────────────────────
n_cb = CB_precip.shape[1]
n_base_const = base_X.shape[1] + 1   # +1 for constant added by fit_qp
coef_cb  = np.asarray(res_inter.params)[n_base_const : n_base_const + n_cb]
vcov_cb  = np.asarray(res_inter.cov_params())[
    n_base_const : n_base_const + n_cb,
    n_base_const : n_base_const + n_cb,
]

# Reference: median non-zero precip
ref_val = np.percentile(nonzero_p, 50)

# Evaluate at percentiles of observed precip
eval_pts = np.percentile(nonzero_p, np.arange(5, 96, 5))

cum_log_rr, cum_se = cumulative_rr(coef_cb, vcov_cb, cb_meta, eval_pts, reference_exp=ref_val)
cum_rr = np.exp(cum_log_rr)
cum_lo = np.exp(cum_log_rr - 1.96 * cum_se)
cum_hi = np.exp(cum_log_rr + 1.96 * cum_se)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(eval_pts, cum_rr, color="steelblue", lw=2, label="Cum. RR")
ax.fill_between(eval_pts, cum_lo, cum_hi, alpha=0.2, color="steelblue")
ax.axhline(1.0, color="black", lw=0.8, ls="--")
ax.axvline(ref_val, color="gray", lw=0.8, ls="--", label=f"ref={ref_val:.1f}mm")
ax.set_xlabel("Daily precipitation (mm)")
ax.set_ylabel("Cumulative RR (lags 0–14)")
ax.set_title("Track A: Precipitation effect on daily diarrheal counts")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "trackA_precip_cumRR.png", dpi=150)
plt.close()
print("Saved: trackA_precip_cumRR.png")


Saved: trackA_precip_cumRR.png


In [8]:
# ─── Track A: screen additional exposures ─────────────────────────────────────

OTHER_VARS = {
    "T_mean_C":      "Daily mean temperature (°C)",
    "T_max_C":       "Daily max temperature (°C)",
    "Td_C":          "Dewpoint temperature (°C)",
    "SM1":           "Surface soil moisture (m³/m³)",
    "wind_speed_ms": "Wind speed (m/s)",
}

print(f"{'Variable':<16} {'F_main':>7} {'p_main':>8} {'F_int':>7} {'p_int':>8}  Sig")
print("-" * 60)

screen_results = []

for varname, desc in OTHER_VARS.items():
    lag_cols = [varname] + [f"{varname}_lag{k}" for k in range(1, 15)]
    if any(c not in df_full.columns for c in lag_cols):
        print(f"  {varname:<14} SKIP (columns missing)")
        continue

    Q = df_full[lag_cols].values
    all_v = Q.flatten()
    zero_f = (all_v == 0).mean()
    nonzero_v = all_v[all_v > 0]
    if len(nonzero_v) == 0:
        continue
    ik = np.percentile(nonzero_v, 80) if zero_f > 0.3 else np.percentile(all_v, 50)
    ik = np.clip(ik, all_v.min() + 1e-6, all_v.max() - 1e-6)

    CB, _, meta_v = build_crossbasis(
        Q, exp_n_int=1, lag_n_int=2,
        exp_all_knots=np.array([all_v.min(), ik, all_v.max()]),
        lag_all_knots=lag_all_knots,
        lag_values=lag_vals,
    )
    CB_xi = CB * infra_c[:, None]

    res_m = fit_qp(np.c_[base_X, CB],       outcome)
    res_i = fit_qp(np.c_[base_X, CB, CB_xi], outcome)

    Fm, pm = ftest(res_base, res_m)
    Fi, pi = ftest(res_m,    res_i)

    sig = "***" if pi < 0.001 else ("**" if pi < 0.01 else ("*" if pi < 0.05 else ""))
    print(f"  {varname:<14} {Fm:>7.3f} {pm:>8.4f} {Fi:>7.3f} {pi:>8.4f}  {sig}")
    screen_results.append(dict(variable=varname, description=desc,
                               F_main=Fm, p_main=pm, F_int=Fi, p_int=pi))

import pandas as pd
screen_df = pd.DataFrame(screen_results).sort_values("p_int")
screen_df.to_csv(OUT_DIR / "trackA_screening.csv", index=False)
print("\nSaved: trackA_screening.csv")


Variable          F_main   p_main   F_int    p_int  Sig
------------------------------------------------------------
  T_mean_C        22.975   0.0000  20.102   0.0000  ***
  T_max_C         20.333   0.0000  20.460   0.0000  ***
  Td_C             4.715   0.0090  16.912   0.0000  ***
  SM1             10.721   0.0000  13.086   0.0000  ***
  wind_speed_ms    2.429   0.0882   5.548   0.0039  **

Saved: trackA_screening.csv


## Track B — Predictive (multi-horizon)

In [9]:
# ─── Track B setup ────────────────────────────────────────────────────────────
# OLS on log(Y+0.5) for computational speed.
# For each horizon h, predict Y_{t+h} from features known at time t.
#
# Models:
#   Seasonal-only:          seasonal spline + DOW + HSA FE + Ramadan + Holiday
#   AR+Seasonal:            + AR(1-day lag) + AR(7-day lag)
#   Seasonal+Climate:       seasonal + climate cross-sum (no AR) — early-warning model
#   AR+Seasonal+Climate:    all three
#
# Climate features: sum over lags 0..14 per variable (parsimony).
# At horizon h, all lags 0..14 at time t are available (past data).

HORIZONS = [1, 7, 14, 21, 28]

# Climate feature matrix: sum of lags 0-14 for each variable (cross-sum approximation)
# This avoids refitting a full cross-basis for each horizon
CLIM_SUM_COLS = {}
for varname in ["P_precip", "T_mean_C", "Td_C", "SM1"]:
    lag_cols = [varname] + [f"{varname}_lag{k}" for k in range(1, 15)]
    if all(c in df_full.columns for c in lag_cols):
        col_sum = df_full[lag_cols].sum(axis=1)
        CLIM_SUM_COLS[varname] = col_sum

clim_matrix = np.column_stack(list(CLIM_SUM_COLS.values())) if CLIM_SUM_COLS else None
print(f"Climate summary features: {list(CLIM_SUM_COLS.keys())}")
print(f"Climate matrix shape: {clim_matrix.shape if clim_matrix is not None else 'None'}")


Climate summary features: ['P_precip', 'T_mean_C', 'Td_C', 'SM1']
Climate matrix shape: (9024, 4)


In [10]:
# ─── Horizon evaluation loop ──────────────────────────────────────────────────

# Temporal split: train on first 80% of dates, test on final 20% across all HSAs
n_obs = len(df_full)
unique_dates = np.array(sorted(df_full["date"].unique()))
split_idx = int(len(unique_dates) * 0.80)
split_date = unique_dates[split_idx]
date_values = df_full["date"].values
print(f"Temporal split date: {pd.Timestamp(split_date).date()}  "
      f"({split_idx}/{len(unique_dates)} dates in training period)")

# Outcomes and AR features per HSA (shift must respect HSA boundaries)
Y_log = np.log(df_full["diarrheal_count"].values + 0.5)

def make_ar_features(h):
    ar1 = df_full.groupby("hsa_id")["diarrheal_count"].shift(1).fillna(0).values
    ar7 = df_full.groupby("hsa_id")["diarrheal_count"].shift(7).fillna(0).values
    return np.c_[np.log(ar1 + 0.5), np.log(ar7 + 0.5)]

def make_future_y(h):
    return df_full.groupby("hsa_id")["diarrheal_count"].shift(-h).values

def r2_oos(y_true, y_pred):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    if len(yt) == 0 or len(yt) != len(yp):
        return np.nan
    ss_res = np.sum((yt - yp)**2)
    ss_tot = np.sum((yt - yt.mean())**2)
    return float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

results_B = []

for h in HORIZONS:
    Y_future = make_future_y(h)
    Y_future_log = np.log(np.where(Y_future > 0, Y_future, 0) + 0.5)
    valid = ~np.isnan(Y_future)

    ar = make_ar_features(h)
    X_seas  = base_X
    X_ar    = np.c_[base_X, ar]
    X_sclim = np.c_[base_X, clim_matrix] if clim_matrix is not None else base_X
    X_full  = np.c_[base_X, ar, clim_matrix] if clim_matrix is not None else X_ar

    train_mask = valid & (date_values < split_date)
    test_mask  = valid & (date_values >= split_date)

    row = {"horizon_days": h}

    for label, X in [
        ("Seasonal-only",            X_seas),
        ("AR+Seasonal",              X_ar),
        ("Seasonal+Climate (no AR)", X_sclim),
        ("AR+Seasonal+Climate",      X_full),
    ]:
        Xc_tr = sm.add_constant(X[train_mask], has_constant="add")
        Xc_te = sm.add_constant(X[test_mask],  has_constant="add")

        try:
            fit = sm.OLS(Y_future_log[train_mask], Xc_tr).fit()
            pred = fit.predict(Xc_te)
            r2 = r2_oos(Y_future_log[test_mask], pred)
        except Exception as e:
            print(f"  WARNING: {label} h={h}d failed: {e}")
            r2 = np.nan

        row[label] = round(r2, 4)

    # Climate contribution over AR baseline
    ar_r2   = row.get("AR+Seasonal", np.nan)
    full_r2 = row.get("AR+Seasonal+Climate", np.nan)
    row["ΔR² (climate)"] = round(full_r2 - ar_r2, 4) if not np.isnan(ar_r2 + full_r2) else np.nan

    results_B.append(row)
    print(f"h={h:2d}d  AR+Seas={ar_r2:.3f}  +Clim={full_r2:.3f}  "
          f"Clim-only={row.get('Seasonal+Climate (no AR)',np.nan):.3f}  "
          f"ΔR²={row['ΔR² (climate)']:.4f}")

results_B_df = pd.DataFrame(results_B)
results_B_df.to_csv(OUT_DIR / "trackB_horizon_results.csv", index=False)
print("\nSaved: trackB_horizon_results.csv")


Temporal split date: 2023-10-11  (451/564 dates in training period)
h= 1d  AR+Seas=-25.089  +Clim=-30.367  Clim-only=-41.752  ΔR²=-5.2784
h= 7d  AR+Seas=-27.693  +Clim=-30.210  Clim-only=-41.608  ΔR²=-2.5166
h=14d  AR+Seas=-11.652  +Clim=-15.009  Clim-only=-20.957  ΔR²=-3.3562
h=21d  AR+Seas=-29.753  +Clim=-34.594  Clim-only=-39.945  ΔR²=-4.8410
h=28d  AR+Seas=-34.070  +Clim=-37.122  Clim-only=-41.493  ΔR²=-3.0518

Saved: trackB_horizon_results.csv


In [11]:
# ─── Track B: horizon plot ────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hors = [r["horizon_days"] for r in results_B]

ax = axes[0]
for model in ["Seasonal-only", "AR+Seasonal", "Seasonal+Climate (no AR)", "AR+Seasonal+Climate"]:
    vals = [r.get(model, np.nan) for r in results_B]
    ax.plot(hors, vals, marker="o", label=model)
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("Out-of-sample R²")
ax.set_title("Track B: Model R² by forecast horizon")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
delta = [r["ΔR² (climate)"] for r in results_B]
delta_plot = [0 if np.isnan(d) else d for d in delta]
colors = ["#9ca3af" if np.isnan(d) else ("#d62728" if d < 0 else "#2ca02c") for d in delta]
ax.bar(hors, delta_plot, color=colors, width=2)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("ΔR² from adding climate to AR+Seasonal")
ax.set_title("Track B: Incremental climate contribution by horizon")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(OUT_DIR / "trackB_horizon_R2.png", dpi=150)
plt.close()
print("Saved: trackB_horizon_R2.png")


Saved: trackB_horizon_R2.png


In [12]:
# ─── Summary table (both tracks) ─────────────────────────────────────────────
print("\n" + "═" * 70)
print("SUMMARY")
print("═" * 70)

print("\nTrack A — Explanatory (DLNM, no AR terms)")
print(f"  Precipitation main effect:  F={F_main:.3f}  p={p_main:.4f}")
print(f"  Sanitation interaction:     F={F_int:.3f}   p={p_int:.4f}")
print(f"  Dispersion φ:              {res_inter.scale:.2f}")

print("\nTrack B — Predictive (multi-horizon OLS on log scale)")
print(results_B_df.to_string(index=False))

print("\nKey finding:")
if p_int < 0.05:
    print("  Track A: Precipitation effect modified by sanitation infrastructure (p<0.05).")
    print("  Climate does affect diarrheal risk, but the magnitude depends on whether")
    print("  sanitation can buffer the exposure-to-contamination pathway.")
else:
    print("  Track A: No significant precipitation effect or sanitation interaction detected.")

valid_deltas = [r["ΔR² (climate)"] for r in results_B if not np.isnan(r["ΔR² (climate)"])]
if valid_deltas:
    max_delta = max(valid_deltas)
    best_h    = [r["horizon_days"] for r in results_B
                 if r["ΔR² (climate)"] == max_delta][0]
    if max_delta > 0:
        print(f"  Track B: Peak climate contribution at h={best_h}d (ΔR²={max_delta:.4f}).")
        print("  Climate adds predictive value where ΔR² is positive;")
        print("  AR terms usually dominate at short horizons.")
    else:
        print(f"  Track B: Climate did not improve holdout R² at any tested horizon; "
              f"least negative ΔR² was h={best_h}d (ΔR²={max_delta:.4f}).")
else:
    print("  Track B: No valid out-of-sample R² values were produced.")



══════════════════════════════════════════════════════════════════════
SUMMARY
══════════════════════════════════════════════════════════════════════

Track A — Explanatory (DLNM, no AR terms)
  Precipitation main effect:  F=1.734  p=0.1767
  Sanitation interaction:     F=5.376   p=0.0046
  Dispersion φ:              3.13

Track B — Predictive (multi-horizon OLS on log scale)
 horizon_days  Seasonal-only  AR+Seasonal  Seasonal+Climate (no AR)  AR+Seasonal+Climate  ΔR² (climate)
            1       -33.6271     -25.0888                  -41.7520             -30.3672        -5.2784
            7       -35.6047     -27.6934                  -41.6075             -30.2100        -2.5166
           14       -15.9160     -11.6525                  -20.9572             -15.0087        -3.3562
           21       -33.6846     -29.7526                  -39.9453             -34.5936        -4.8410
           28       -36.9931     -34.0701                  -41.4932             -37.1219        -3.0

In [13]:
# AUTO_NOTEBOOK_SUMMARY_V1
from pathlib import Path
from datetime import datetime
import os
import json

NOTEBOOK_NAME = "run_climate_models_daily"
NETWORK = globals().get('NETWORK', os.environ.get('NETWORK', 'INF'))
HSA_MODE = globals().get('HSA_MODE', os.environ.get('HSA_MODE', 'footprint'))

suffix = f"{NETWORK}_{HSA_MODE}" if HSA_MODE else f"{NETWORK}"

out_root = Path(globals().get('OUT_ROOT', globals().get('OUT_DIR', os.environ.get('HSA_OUT_DIR', os.environ.get('PIPELINE_OUT_DIR', "out")))))
# OUT_DIR in this notebook points inside modeling/daily_models_{ver}; walk up to the pipeline root
if 'modeling' in str(out_root):
    out_root = out_root.parent.parent
summary_dir = out_root / 'textresults'
summary_dir.mkdir(parents=True, exist_ok=True)
BOUNDARY_VERSION = globals().get('BOUNDARY_VERSION', os.environ.get('BOUNDARY_VERSION', os.environ.get('PIPELINE_VERSION', 'v7')))
summary_path = summary_dir / f"{NOTEBOOK_NAME}_{suffix}_{BOUNDARY_VERSION}_results.md"

files = [p for p in out_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.md', '.txt', '.png', '.pdf', '.geojson', '.parquet'}]
files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
important = files[:120]

NL = "\n"

blocks = []
blocks.append(f"# Notebook Results: {NOTEBOOK_NAME}")

meta = [
    f"- Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"- Network: {NETWORK}",
    f"- HSA mode: {HSA_MODE}",
    f"- Boundary version: {BOUNDARY_VERSION}",
]
blocks.append(NL.join(meta))

file_lines = ['## Important Output Files', '']
for p in important:
    file_lines.append(f"- `{p}`")
blocks.append(NL.join(file_lines))

nb_path = Path(f"{NOTEBOOK_NAME}.ipynb")
if nb_path.exists():
    try:
        nb_data = json.loads(nb_path.read_text())
        blocks.append('## Displayed Cell Outputs')

        for idx, cell in enumerate(nb_data.get('cells', []), start=1):
            if cell.get('cell_type') != 'code':
                continue
            outputs = cell.get('outputs', []) or []
            if not outputs:
                continue

            blocks.append(f"### Cell {idx}")

            for out in outputs:
                otype = out.get('output_type')

                if otype == 'stream':
                    text = ''.join(out.get('text', [])) if isinstance(out.get('text', []), list) else str(out.get('text', ''))
                    text = text.rstrip()
                    if text:
                        blocks.append("```text" + NL + text + NL + "```")

                elif otype in ('execute_result', 'display_data'):
                    data = out.get('data', {})
                    if 'text/markdown' in data:
                        md = ''.join(data['text/markdown']) if isinstance(data['text/markdown'], list) else str(data['text/markdown'])
                        md = md.rstrip()
                        if md:
                            blocks.append(md)
                    elif 'text/plain' in data:
                        txt = ''.join(data['text/plain']) if isinstance(data['text/plain'], list) else str(data['text/plain'])
                        txt = txt.rstrip()
                        if txt:
                            blocks.append("```text" + NL + txt + NL + "```")
                    elif 'text/html' in data:
                        html = ''.join(data['text/html']) if isinstance(data['text/html'], list) else str(data['text/html'])
                        html = html.rstrip()
                        if html:
                            blocks.append("```html" + NL + html + NL + "```")
                    elif 'image/png' in data or 'image/jpeg' in data:
                        blocks.append('_[Image output omitted in text summary]_')

                elif otype == 'error':
                    tb = out.get('traceback', []) or []
                    if tb:
                        err = NL.join(str(t) for t in tb)
                    else:
                        err = f"{out.get('ename', 'Error')}: {out.get('evalue', '')}"
                    blocks.append("```text" + NL + err + NL + "```")

    except Exception as e:
        blocks.append('## Displayed Cell Outputs')
        blocks.append(f"Could not parse notebook outputs: {e}")

summary = (NL + NL).join(b for b in blocks if b and str(b).strip()) + NL
summary_path.write_text(summary)
print(f"Saved notebook summary: {summary_path}")


Saved notebook summary: out/textresults/run_climate_models_daily_INF_footprint_v8_results.md
